# 정적 웹페이지 수집하기
* 순수 HTML, CSS로 만들어진 페이지
* javascript로 내용을 갱신하지 않는 페이지

In [ ]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

# yes24 베스트셀러 자료 수집하기

In [ ]:
url="https://www.yes24.com/product/category/bestseller"
payload=dict(categoryNumber="001", pageNumber=1, pageSize=120)
r=requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup=bs(r.content,'lxml')
soup

* 전체 yes24 베스트셀러 페이지 중 책 정보가 들어있는 곳: ul#yesBestList
* ul#yesBestList 아래의 li에 책 1권 정보가 들어있음

In [ ]:
book_list=soup.select("ul#yesBestList > li")

In [ ]:
result={}
for idx, book in enumerate(book_list):
    print(f"{idx+1}/{len(book_list)} 추출중", end="\r")
    #책 제목
    book_title=book.select_one('.gd_name').text
    #저자
    author=book.select_one(".info_row.info_pubGrp a").text
    #출판사
    publisher=book.select_one('.authPub.info_pub a').text
    #출간일
    date_pub=book.select_one(".authPub.info_date").text
    #가격
    price=book.select_one(".info_row.info_price em.yes_b").text
    #회원리뷰 점수
    rating=book.select_one(".rating_grade > em.yes_b").text if book.select_one(".rating_grade > em.yes_b")!=None else 0.0
    #회원리뷰수
    n_reviews=book.select_one(".rating_rvCount em.txC_blue").text if book.select_one(".rating_rvCount em.txC_blue")!=None else 0
    
    keys=['book_title','author','publisher','date_pub','price','rating','n_reviews']
    values=[book_title, author, publisher, date_pub, price, rating, n_reviews]
    for key, value in zip(keys, values): 
        result.setdefault(key,[]).append(value)

for key, value in result.items():
    print(key, len(value))
    
df=pd.DataFrame(result)
df

In [ ]:
data = []

for idx, book in enumerate(book_list):
    print(f"{idx+1}/{len(book_list)} 추출중", end="\r")
    
    data.append({
        'book_title': book.select_one('.gd_name').text,
        'author': book.select_one('.info_row.info_pubGrp a').text,
        'publisher': book.select_one('.authPub.info_pub a').text,
        'date_pub': book.select_one('.authPub.info_date').text,
        'price': book.select_one('.info_row.info_price em.yes_b').text,
        'rating': book.select_one('.rating_grade > em.yes_b').text if book.select_one('.rating_grade > em.yes_b') else 0.0,
        'n_reviews': book.select_one('.rating_rvCount em.txC_blue').text if book.select_one('.rating_rvCount em.txC_blue') else 0,
    })

df = pd.DataFrame(data)
df

In [ ]:
print('가격: ', len(list(soup.select("ul#yesBestList .info_row.info_price em.yes_b"))))
print("평점: ", len(list(soup.select("ul#yesBestList .rating_grade > em.yes_b"))))
print("리뷰수: ", len(list(soup.select("ul#yesBestList .rating_rvCount em.txC_blue" ))))

In [ ]:
link="https://www.yes24.com"+book_list[20].select_one(".gd_name")['href']

In [ ]:
result['author']

In [ ]:
book_list[10].select_one(".info_row.info_pubGrp").text

In [ ]:
book_list[10].select_one(".authPub.info_auth").text.split("/")[0]

# 전체 페이지 수집하기

In [ ]:
page=1
data = []
while True:
    url="https://www.yes24.com/product/category/bestseller"
    payload=dict(categoryNumber="001", pageNumber=page, pageSize=120)
    r=requests.get(url, params=payload)
    print(r.url)
    print(r.status_code)
    soup=bs(r.content,'lxml')
    time.sleep(5)

    
    print(f"{page}페이지 추출중", end="\r")
    
    for idx, book in enumerate(book_list):
        print(f"{idx+1}/{len(book_list)} 추출중", end="\r")
        
        data.append({
            'book_title': book.select_one('.gd_name').text,
            'author': book.select_one('.info_row.info_pubGrp a').text,
            'publisher': book.select_one('.authPub.info_pub a').text,
            'date_pub': book.select_one('.authPub.info_date').text,
            'price': book.select_one('.info_row.info_price em.yes_b').text,
            'rating': book.select_one('.rating_grade > em.yes_b').text if book.select_one('.rating_grade > em.yes_b') else 0.0,
            'n_reviews': book.select_one('.rating_rvCount em.txC_blue').text if book.select_one('.rating_rvCount em.txC_blue') else 0,
        })
    
    if page<9:
        page+=1
    else:
        break

df = pd.DataFrame(data)
df

# 저자, 역자, 글그림, 편저, 공동저자 등 구분하기

In [1]:
def text_clean(text):
    return text.replace('\n','').replace('\r','').strip()

In [3]:
def author_extraction(book):
    author=''
    photo=''
    trans=''
    paint=''
    for idx, item in enumerate(text_clean(book.select_one(".authPub.info_auth").text).split("/")):
        print(idx, item)
        if idx==0:
            if '저'==item[-1]:
                author=text_clean(item[:-2])
            elif '글'==item[-1]:
                author=text_clean(item[:-2])
            elif '글그림'==item[-3:]:
                author=text_clean(item[:-4])
        else:
            if '정보 더 보기' == item[-7:]:
                author=text_clean(item.replace('정보 더보기',""))
            elif '사진'==item[-2:]:
                photo=text_clean(item[:-3])
            elif '역'==item[-1]:
                trans=text_clean(item[:-2])
            elif '글그림'==item[-3:]:
                paint=text_clean(item[:-4])
    print(f"author{author}, photo{photo}, trans{trans}, paint{paint}")
    return author, photo, trans, paint

In [4]:
url="https://www.yes24.com/product/category/bestseller"
payload=dict(categoryNumber="001", pageNumber=1, pageSize=120)
r=requests.get(url, params=payload)
print(r.url)
print(r.status_code)
soup=bs(r.content,'lxml')
time.sleep(5)
book_list=soup.select("ul#yesBestList > li")
for book in book_list[:2]:
    author, photo, trans, paint=author_extraction(book)
    print(author, photo, trans, paint)

NameError: name 'requests' is not defined

In [5]:
for item in text_clean(book_list[20].select_one(".authPub.info_auth").text).split("/"):
#     print(item)
    if "감추기" in item[:4]:
        author = item.replace("감추기", "").strip()
        print(author)

NameError: name 'book_list' is not defined